In [3]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [4]:
!pip install -q vllm openai
!pip uninstall -y -q torchaudio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
aiobotocore 2.26.0 requires multidict<7.0.0,>=6.0.0, but you have multidict 7.2.1 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.2.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
pylibcudf-cu12 26.2.1 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.4.4 which is incompatible.
numba-cuda 0.22.2 requires cuda-core<1.0.0,>=0.3.2, but you have cuda-core 1.2.2 which is incompatible.
libcuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-to

In [5]:
import subprocess

def ver(pkg):
    r = subprocess.run(["python", "-c", f"import {pkg}; print({pkg}.__version__)"],
                       capture_output=True, text=True)
    return r.stdout.strip() or "IMPORT FAILED: " + r.stderr.strip()[-200:]

def suffix(v):
    return v.split("+")[1] if "+" in v else ""

torch_v, tv_v = ver("torch"), ver("torchvision")
print("vllm", ver("vllm"), "| torch", torch_v, "| torchvision", tv_v)

if suffix(torch_v).startswith("cu") and suffix(tv_v) != suffix(torch_v):
    cu = suffix(torch_v)
    print(f"torchvision does not match torch -> reinstalling torchvision for {cu} ...")
    subprocess.run(["pip", "install", "-q", "--force-reinstall", "--no-deps", "torchvision",
                    "--index-url", f"https://download.pytorch.org/whl/{cu}"])
    print("now: torch", ver("torch"), "| torchvision", ver("torchvision"))

vllm 0.33.0 | torch 2.14.0+cu130 | torchvision 0.29.0+cu130


In [6]:
import subprocess, time, requests

MODEL = "Qwen/Qwen2.5-Coder-14B-Instruct-AWQ"   # fallback if it fails: "Qwen/Qwen2.5-Coder-7B-Instruct"

log = open("vllm.log", "w")
proc = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL,
     "--dtype", "half", "--tensor-parallel-size", "2", "--max-model-len", "16384",
     "--gpu-memory-utilization", "0.90", "--port", "8000"],
    stdout=log, stderr=subprocess.STDOUT)

for i in range(120):
    if proc.poll() is not None:
        print("SERVER CRASHED. Last part of the log:\n")
        print(open("vllm.log").read()[-3000:])
        break
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
            print("✅ server up after", i * 5, "seconds")
            break
    except Exception:
        pass
    time.sleep(5)
else:
    print("Still not up after 10 min. Last part of the log:\n")
    print(open("vllm.log").read()[-3000:])

✅ server up after 290 seconds


In [7]:
import time, statistics
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI

c = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
helper = "static int helper_%d(char *p, int n) { if (n > 0) memcpy(p, p + 1, n); return n; }\n"
ctx = ("// === TARGET FUNCTION (a.c:1-5) ===\n"
       "int f(char *s){char b[8]; strcpy(b,s); return 0;}\n"
       + "".join(helper % i for i in range(450)))
msgs = [{"role": "user", "content": ctx + '\nIs the TARGET FUNCTION vulnerable? Reply with JSON '
         '{"vulnerable": bool, "cwe": str|null, "line": int|null, "confidence": float}'}]

def one(_):
    t = time.time()
    r = c.chat.completions.create(model=MODEL, messages=msgs, max_tokens=200, temperature=0.7)
    return time.time() - t, r.usage.prompt_tokens, r.usage.completion_tokens, r.choices[0].message.content

print("sample reply:", one(0)[3][:200], "\n")
for workers, n in ((1, 5), (4, 20)):
    t0 = time.time()
    with ThreadPoolExecutor(workers) as ex:
        res = list(ex.map(one, range(n)))
    wall = time.time() - t0
    lat = [r[0] for r in res]
    tin = sum(r[1] for r in res)
    print(f"workers={workers} n={n} prompt_tokens≈{res[0][1]} "
          f"p50={statistics.median(lat):.1f}s max={max(lat):.1f}s "
          f"throughput={tin / wall:.0f} input tok/s")

!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

sample reply: ```json
{
  "vulnerable": false,
  "cwe": null,
  "line": null,
  "confidence": 1.0
}
``` 

workers=1 n=5 prompt_tokens≈16185 p50=11.5s max=14.1s throughput=1577 input tok/s
workers=4 n=20 prompt_tokens≈16185 p50=7.8s max=37.0s throughput=3725 input tok/s
memory.used [MiB], memory.total [MiB]
13807 MiB, 15360 MiB
13807 MiB, 15360 MiB


In [8]:
proc.terminate()
print("stopped")

stopped
